# 第16章 速く・大きく：推論と学習の工学

KVキャッシュ、賢いサイコロ、メモリの節約術、そして何百枚ものGPU

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/16-scale.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。


## 準備：キャッシュ対応のGPT


In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F

class CachedAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)

    def forward(self, x, cache=None):
        B, T, C = x.shape
        hs = C // self.n_head
        q, k, v = self.c_attn(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.n_head, hs).transpose(1, 2) for t in (q, k, v)]
        if cache is not None:                      # 過去のKとVを使い回す
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        L = k.shape[2]                             # 過去も含めた長さ
        att = q @ k.transpose(-2, -1) / math.sqrt(hs)
        mask = torch.tril(torch.ones(L, L))[L - T:]    # 新しいT個の行だけのマスク
        att = att.masked_fill(mask == 0, float("-inf"))
        y = F.softmax(att, dim=-1) @ v
        return self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C))

class TinyGPT(nn.Module):
    def __init__(self, V=1000, n_embd=256, n_head=4, n_layer=6, block_size=512):
        super().__init__()
        self.wte, self.wpe = nn.Embedding(V, n_embd), nn.Embedding(block_size, n_embd)
        self.ln1 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layer)])
        self.attn = nn.ModuleList([CachedAttention(n_embd, n_head) for _ in range(n_layer)])
        self.ln2 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layer)])
        self.mlp = nn.ModuleList([nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd)) for _ in range(n_layer)])
        self.ln_f, self.head = nn.LayerNorm(n_embd), nn.Linear(n_embd, V)

    def forward(self, idx, caches=None, start_pos=0):
        T = idx.shape[1]
        x = self.wte(idx) + self.wpe(torch.arange(start_pos, start_pos + T))
        for i in range(len(self.attn)):
            x = x + self.attn[i](self.ln1[i](x), None if caches is None else caches[i])
            x = x + self.mlp[i](self.ln2[i](x))
        return self.head(self.ln_f(x))

torch.manual_seed(0)
model = TinyGPT().eval()
print(f"パラメータ：{sum(p.numel() for p in model.parameters()):,}")

## ムダに気づく：生成のたびに全部計算し直している


In [ ]:
@torch.no_grad()
def generate_naive(model, idx, n):
    for _ in range(n):
        logits = model(idx)                              # 毎回、全体を計算
        idx = torch.cat([idx, logits[:, -1:].argmax(-1)], dim=1)
    return idx

@torch.no_grad()
def generate_cached(model, idx, n):
    caches = [{} for _ in model.attn]
    logits = model(idx, caches)                          # 最初だけ全体を計算
    pos = idx.shape[1]
    for _ in range(n):
        next_id = logits[:, -1:].argmax(-1)
        idx = torch.cat([idx, next_id], dim=1)
        logits = model(next_id, caches, start_pos=pos)   # 新しい1トークンだけ
        pos += 1
    return idx

prompt = torch.randint(0, 1000, (1, 20))
# time.perf_counter() は「今の時刻（秒）」。前後の差が、かかった時間になる
t0 = time.perf_counter(); a = generate_naive(model, prompt, 200); t1 = time.perf_counter()
b = generate_cached(model, prompt, 200); t2 = time.perf_counter()
print("結果は同じ？", torch.equal(a, b))
print(f"キャッシュなし {t1 - t0:.2f}秒 / キャッシュあり {t2 - t1:.2f}秒 → {(t1 - t0) / (t2 - t1):.1f}倍速い")

## KVキャッシュのメモリと、GQA


In [ ]:
def kv_cache_bytes(n_layer, n_kv_head, head_dim, n_tokens, bytes_per_num=2):
    return 2 * n_layer * n_kv_head * head_dim * n_tokens * bytes_per_num   # 2 = KとV

# Qwen2.5-0.5B：24層、Queryのヘッド14、K・Vのヘッド2、ヘッドの大きさ64
per_token = kv_cache_bytes(24, 2, 64, 1)
print(f"1トークンあたり {per_token / 1024:.0f} KB → 32,768トークンで {kv_cache_bytes(24, 2, 64, 32768) / 1e9:.2f} GB")
print(f"もしK・Vのヘッドも14個なら → 32,768トークンで {kv_cache_bytes(24, 14, 64, 32768) / 1e9:.2f} GB")

## 賢いサイコロ：top-k と top-p


In [ ]:
words = ["猫", "犬", "人間", "名前", "机", "宇宙", "の"]
logits = torch.tensor([3.0, 1.6, 1.2, 0.4, -1.0, -1.5, -2.0])
probs = F.softmax(logits, dim=0)

def top_k_filter(probs, k):
    keep = torch.zeros_like(probs)
    idx = torch.topk(probs, k).indices
    keep[idx] = probs[idx]
    return keep / keep.sum()

def top_p_filter(probs, p):
    sorted_p, order = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_p, dim=0)
    n_keep = int((cum < p).sum()) + 1            # 累積がpを超えるまでの最小の候補数
    keep = torch.zeros_like(probs)
    keep[order[:n_keep]] = probs[order[:n_keep]]
    return keep / keep.sum()

show = lambda p: "  ".join(f"{w}{x:.2f}" for w, x in zip(words, p) if x > 0)
print("そのまま ：", show(probs))
print("top-k=3 ：", show(top_k_filter(probs, 3)))
print("top-p=0.9：", show(top_p_filter(probs, 0.9)))

## 学習の工学1：勾配累積


In [ ]:
torch.manual_seed(0)
net = nn.Linear(16, 4)
x, y = torch.randn(32, 16), torch.randint(0, 4, (32,))

net.zero_grad()
F.cross_entropy(net(x), y).backward()            # バッチ32で1回
g_big = net.weight.grad.clone()

net.zero_grad()
for i in range(4):                               # バッチ8を4回に分ける
    xb, yb = x[i * 8:(i + 1) * 8], y[i * 8:(i + 1) * 8]
    (F.cross_entropy(net(xb), yb) / 4).backward()    # 4で割ってから足し込む
g_acc = net.weight.grad

print("勾配は同じ？", torch.allclose(g_big, g_acc, atol=1e-6))

## 学習の工学2：活性化の再計算


In [ ]:
from torch.utils.checkpoint import checkpoint

block = nn.Sequential(nn.Linear(64, 256), nn.GELU(), nn.Linear(256, 64))
x = torch.randn(8, 64, requires_grad=True)
out1 = block(x).sum(); out1.backward(); g1 = x.grad.clone()
x.grad = None
out2 = checkpoint(block, x, use_reentrant=False).sum(); out2.backward(); g2 = x.grad
print("同じ勾配？", torch.allclose(g1, g2))

## 学習の工学3：FlashAttention


In [ ]:
torch.manual_seed(0)
q, k, v = (torch.randn(2, 4, 128, 32) for _ in range(3))     # [B, ヘッド, T, hs]

att = q @ k.transpose(-2, -1) / math.sqrt(32)
att = att.masked_fill(torch.tril(torch.ones(128, 128)) == 0, float("-inf"))
manual = F.softmax(att, dim=-1) @ v

fast = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("手作りと同じ？", torch.allclose(manual, fast, atol=1e-5))